# Практическая работа №4: Временные ряды и пользовательские сессии
**Вариант 4**

Задания варианта:
1. Определить сессии пользователей с таймаутом 30 минут.
2. Для каждой сессии посчитать длительность и число событий.
3. Найти 10 самых длинных сессий по длительности.

In [1]:
import pandas as pd
import numpy as np

## Загрузка событий

In [2]:
df_events_records = pd.read_json("data/events_records.json").T.reset_index(drop=True)
df_events_records.columns = ["event_id", "user_id", "event_time", "event_type", "page"]

df_events_lines = pd.read_json("data/events_lines.json", lines=True)

df_events = pd.concat([df_events_records, df_events_lines], ignore_index=True)
df_events = df_events.astype({"event_id": "int64", "user_id": "int64", "event_type": "string", "page": "string"})
df_events["event_time"] = pd.to_datetime(df_events["event_time"])

print(df_events.head())

   event_id  user_id          event_time   event_type         page
0         1     1085 2024-01-30 20:37:32    page_view  /product/26
1         2     1006 2024-02-24 22:55:13  add_to_cart  /product/28
2         3     1024 2024-02-14 17:49:43        click  /product/27
3         4     1062 2024-02-16 11:21:32        click   /product/1
4         5     1012 2024-02-08 05:25:21  add_to_cart            /


## Задание 1. Определение сессий с таймаутом 30 минут

In [3]:
df_events_user = df_events.sort_values(["user_id", "event_time"]).reset_index(drop=True)

df_events_user["prev_event_time"] = df_events_user.groupby("user_id")["event_time"].shift(1)
df_events_user["delta_since_prev"] = df_events_user["event_time"] - df_events_user["prev_event_time"]

session_timeout = pd.Timedelta(minutes=30)

df_events_user["is_new_session"] = (
    df_events_user["delta_since_prev"].isna()
    | (df_events_user["delta_since_prev"] > session_timeout)
)

df_events_user["session_id"] = df_events_user.groupby("user_id")["is_new_session"].cumsum()

print(df_events_user[["user_id", "event_time", "delta_since_prev", "is_new_session", "session_id"]].head(15))

    user_id          event_time delta_since_prev  is_new_session  session_id
0      1001 2024-01-01 06:25:31              NaT            True           1
1      1001 2024-01-05 07:12:56  4 days 00:47:25            True           2
2      1001 2024-01-08 15:40:51  3 days 08:27:55            True           3
3      1001 2024-01-09 06:39:26  0 days 14:58:35            True           4
4      1001 2024-01-15 00:00:38  5 days 17:21:12            True           5
5      1001 2024-01-20 00:29:26  5 days 00:28:48            True           6
6      1001 2024-01-24 08:41:28  4 days 08:12:02            True           7
7      1001 2024-01-24 11:54:49  0 days 03:13:21            True           8
8      1001 2024-01-25 17:54:45  1 days 05:59:56            True           9
9      1001 2024-01-26 17:20:09  0 days 23:25:24            True          10
10     1001 2024-01-28 09:17:41  1 days 15:57:32            True          11
11     1001 2024-01-28 20:00:11  0 days 10:42:30            True          12

## Задание 2. Длительность и число событий в каждой сессии

In [4]:
session_stats = (
    df_events_user
    .groupby(["user_id", "session_id"])
    .agg(
        session_start=("event_time", "min"),
        session_end=("event_time", "max"),
        events_in_session=("event_id", "count"),
    )
    .reset_index()
)

session_stats["session_duration"] = session_stats["session_end"] - session_stats["session_start"]

print(session_stats.head(10))

   user_id  session_id       session_start         session_end  \
0     1001           1 2024-01-01 06:25:31 2024-01-01 06:25:31   
1     1001           2 2024-01-05 07:12:56 2024-01-05 07:12:56   
2     1001           3 2024-01-08 15:40:51 2024-01-08 15:40:51   
3     1001           4 2024-01-09 06:39:26 2024-01-09 06:39:26   
4     1001           5 2024-01-15 00:00:38 2024-01-15 00:00:38   
5     1001           6 2024-01-20 00:29:26 2024-01-20 00:29:26   
6     1001           7 2024-01-24 08:41:28 2024-01-24 08:41:28   
7     1001           8 2024-01-24 11:54:49 2024-01-24 11:54:49   
8     1001           9 2024-01-25 17:54:45 2024-01-25 17:54:45   
9     1001          10 2024-01-26 17:20:09 2024-01-26 17:20:09   

   events_in_session session_duration  
0                  1           0 days  
1                  1           0 days  
2                  1           0 days  
3                  1           0 days  
4                  1           0 days  
5                  1           0 

## Задание 3. 10 самых длинных сессий по длительности

In [5]:
top10_sessions = session_stats.sort_values("session_duration", ascending=False).head(10)

print(top10_sessions)

      user_id  session_id       session_start         session_end  \
35       1001          36 2024-02-22 15:17:40 2024-02-22 15:51:20   
1247     1033          29 2024-02-27 18:57:09 2024-02-27 19:27:01   
2388     1061          26 2024-02-17 12:36:27 2024-02-17 13:05:43   
1116     1030          16 2024-02-04 06:30:36 2024-02-04 06:59:51   
1507     1040          13 2024-02-12 17:42:18 2024-02-12 18:11:31   
559      1015           5 2024-01-16 20:16:16 2024-01-16 20:44:19   
979      1026          37 2024-02-26 10:13:24 2024-02-26 10:40:00   
3766     1096          23 2024-02-22 12:17:35 2024-02-22 12:43:26   
1912     1050          29 2024-03-03 12:51:04 2024-03-03 13:15:54   
1506     1040          12 2024-02-12 12:43:58 2024-02-12 13:07:14   

      events_in_session session_duration  
35                    3  0 days 00:33:40  
1247                  2  0 days 00:29:52  
2388                  2  0 days 00:29:16  
1116                  2  0 days 00:29:15  
1507                  3  